# Convert EPS to PNG


In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import argparse
import shutil
import subprocess
import sys
import tempfile

# python kernel selection
# Cmd + P
# input: >Python: Select Interpreter

def find_ghostscript() -> str:
    """Find the Ghostscript executable on macOS/Linux/Windows."""
    candidates = [
        shutil.which("gs"),
        shutil.which("gswin64c"),
        shutil.which("gswin32c"),
        "/opt/homebrew/bin/gs",                 # Apple Silicon Homebrew
        "/usr/local/bin/gs",                    # Intel Homebrew
        "C:/Program Files/gs/gs10.04.0/bin/gswin64c.exe",
    ]

    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(candidate)

    raise FileNotFoundError(
        "Cannot find Ghostscript. Install Ghostscript first, or add it to PATH."
    )

def save_png_uncompressed(input_png: Path, output_png: Path) -> None:
    """Save PNG with compression level 0."""
    try:
        from PIL import Image
    except ImportError as exc:
        raise ImportError(
            "Pillow is required for uncompressed PNG output. Install it with: pip install pillow"
        ) from exc

    with Image.open(input_png) as image:
        image.save(
            output_png,
            format="PNG",
            compress_level=0,
            optimize=False,
        )

def convert_eps_to_png(
    eps_path: Path,
    overwrite: bool = False,
    dpi: int = 300,
    transparent: bool = False,
) -> None:
    eps_path = eps_path.resolve()
    png_path = eps_path.with_suffix(".png")

    if png_path.exists() and not overwrite:
        print(f"Skip existing: {png_path}")
        return

    ghostscript = find_ghostscript()

    device = "pngalpha" if transparent else "png16m"

    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as temp_file:
        temp_png_path = Path(temp_file.name)

    command = [
        ghostscript,
        "-dSAFER",
        "-dBATCH",
        "-dNOPAUSE",
        "-dEPSCrop",
        f"-r{dpi}",
        f"-sDEVICE={device}",
        "-dTextAlphaBits=4",
        "-dGraphicsAlphaBits=4",
        f"-sOutputFile={temp_png_path}",
        str(eps_path),
    ]

    print(f"Converting: {eps_path} -> {png_path}")
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        temp_png_path.unlink(missing_ok=True)
        print(result.stderr)
        raise RuntimeError(f"Failed to convert {eps_path}")

    save_png_uncompressed(temp_png_path, png_path)
    temp_png_path.unlink(missing_ok=True)

    print(f"Done: {png_path}")

def collect_eps_files(path: Path, recursive: bool = False) -> list[Path]:
    path = path.resolve()

    if path.is_file():
        if path.suffix.lower() != ".eps":
            raise ValueError(f"Input file is not an EPS file: {path}")
        return [path]

    if path.is_dir():
        pattern = "**/*.eps" if recursive else "*.eps"
        return sorted(path.glob(pattern))

    raise FileNotFoundError(f"Path does not exist: {path}")

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Convert EPS files to uncompressed PNG using Ghostscript."
    )
    parser.add_argument(
        "path",
        nargs="?",
        default=".",
        help="EPS file or directory containing EPS files. Default: current directory.",
    )
    parser.add_argument(
        "-r", "--recursive",
        action="store_true",
        help="Search EPS files recursively in subdirectories.",
    )
    parser.add_argument(
        "-f", "--overwrite",
        action="store_true",
        help="Overwrite existing PNG files.",
    )
    parser.add_argument(
        "-d", "--dpi",
        type=int,
        default=300,
        help="Rasterization resolution in DPI. Default: 300.",
    )
    parser.add_argument(
        "--transparent",
        action="store_true",
        help="Use transparent background instead of white background.",
    )

    args, unknown = parser.parse_known_args()

    eps_files = collect_eps_files(Path(args.path), recursive=args.recursive)

    if not eps_files:
        print("No EPS files found.")
        return

    for eps_file in eps_files:
        convert_eps_to_png(
            eps_file,
            overwrite=args.overwrite,
            dpi=args.dpi,
            transparent=args.transparent,
        )

if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"Error: {exc}", file=sys.stderr)
        sys.exit(1)


Converting: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.6.eps -> /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.6.png
Done: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.6.png
Converting: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.8.eps -> /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.8.png
Done: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.8.png
